# 01 — Data Quality
Sanity-check the option chain data before trusting any backtest built on it:
coverage, missing days, quote sanity, and delta/IV consistency.

### Load the full chain dataset
Loads all parquets from 2010–2023. Examine coverage, quote quality, price path.


In [ ]:
import sys
sys.path.insert(0, "../src")
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11, 4)
pd.set_option("display.width", 160)

### Monthly coverage check
Expect ~21 trading days per month. <18 days = suspect data.


In [ ]:
from lab.backtest import load_chains

chains = load_chains()   # full 2010-2023; pass start/end for a faster slice
print(f"{len(chains):,} contract rows, {chains.quote_date.min():%Y-%m-%d} to {chains.quote_date.max():%Y-%m-%d}")
chains.head()

## Coverage: contracts per month and trading-day gaps

### Quote sanity
Crossed markets (bid > ask), negative bids, delta signs, spreads.


In [ ]:
per_month = chains.groupby(chains.quote_date.dt.to_period("M")).size()
per_month.plot(title="Contract rows per month");

### SPX price path
Underlying price from the chains—if wrong, data is bad.


In [ ]:
# Trading days present vs NYSE-ish expectation (~21/month)
days = chains.groupby(chains.quote_date.dt.to_period("M"))["quote_date"].nunique()
suspect = days[days < 18]
print("Months with < 18 quote days (investigate):")
print(suspect if len(suspect) else "none")

## Quote sanity: bid/ask and delta distributions

In [ ]:
sample = chains.sample(min(500_000, len(chains)), random_state=0)
print("crossed markets (bid > ask):", (sample.bid > sample.ask).mean().round(5))
print("negative bids:", (sample.bid < 0).mean().round(5))
sample["spread"] = sample.ask - sample.bid
print(sample.groupby("option_type")[["bid", "ask", "spread", "delta"]].describe().round(3).T)

In [ ]:
# Put deltas must be negative, call deltas positive
bad_delta = ((sample.option_type == "p") & (sample.delta > 0)) | ((sample.option_type == "c") & (sample.delta < 0))
print("delta sign violations:", bad_delta.mean().round(5))
sample.hist(column="delta", by="option_type", bins=60);

## Underlying price path from the chains

In [ ]:
spx = chains.groupby("quote_date")["underlying_price"].last()
spx.plot(title="SPX from chain underlying_price");